# grad-tracking-global-toggle — faded example 3: Faded: SetGradEnabled __exit__ restores previous flag

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-tracking-global-toggle`. Running the beacon reports progress on the `Backprop: Grad-tracking toggle` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad-tracking toggle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-tracking-global-toggle`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-tracking-global-toggle"
DD_SUBTOPIC = "Backprop: Grad-tracking toggle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The exit method of a grad-tracking context manager is responsible for the snapshot-restore step: whatever the flag was before `__enter__` ran must be written back when `__exit__` runs. This is what makes the pattern nesting-safe — `__exit__` does not hard-code `True` or `False`, it uses the `self.prev` snapshot stored during `__enter__`. Returning `False` from `__exit__` allows any exception raised in the `with` block to propagate normally.

## Faded exercise 3

Complete `SetGradEnabled.__exit__`. The blank restores the previously snapshotted flag value and ensures exceptions are not suppressed.

**Fill in:** The two-line body of __exit__: restore the global flag to self.prev, then return False.

In [ ]:
grad_tracking_enabled = True

def _get_flag():
    return globals()['grad_tracking_enabled']

def _set_flag(v):
    globals()['grad_tracking_enabled'] = v

class SetGradEnabled:
    def __init__(self, mode):
        self.mode = bool(mode)
        self.prev = None
    def __enter__(self):
        self.prev = _get_flag()
        _set_flag(self.mode)
        return self
    def __exit__(self, exc_type, exc_val, tb):
        _set_flag(self.prev)
        return False

# Exercise it
with SetGradEnabled(False):
    print(_get_flag())  # False
print(_get_flag())  # True


def _test():
    # Basic restore True->False->True
    assert _get_flag() == True
    with SetGradEnabled(False):
        assert _get_flag() == False
    assert _get_flag() == True

    # Restore to False when already False (nested)
    with SetGradEnabled(False):
        with SetGradEnabled(True):
            assert _get_flag() == True
        assert _get_flag() == False, "Exit must restore to False (the outer's prior value)"
    assert _get_flag() == True

    # Exception safety: exception propagates, flag still restored
    try:
        with SetGradEnabled(False):
            raise ValueError("test")
    except ValueError:
        pass
    assert _get_flag() == True, "Flag must be restored even when exception leaves the with block"


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
grad_tracking_enabled = True

def _get_flag():
    return globals()['grad_tracking_enabled']

def _set_flag(v):
    globals()['grad_tracking_enabled'] = v

class SetGradEnabled:
    def __init__(self, mode):
        self.mode = bool(mode)
        self.prev = None
    def __enter__(self):
        self.prev = _get_flag()
        _set_flag(self.mode)
        return self
    def __exit__(self, exc_type, exc_val, tb):
        _set_flag(self.prev)
        return False

# Exercise it
with SetGradEnabled(False):
    print(_get_flag())  # False
print(_get_flag())  # True
```
</details>